# 🧪 Notebook 07 — Experiments, Evaluation & Stress Test
**NB6057CEM CW1 | T4 (14 marks)**

Aggregates results from all seeds, produces the full metrics table, error analysis, robustness stress test, and final comparison between all 3 models.

---

In [ ]:
import torch, torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json, os, csv, time
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from google.colab import drive

drive.mount('/content/drive')

# Restore splits and preprocessing from Drive if not found locally
os.makedirs('splits', exist_ok=True)
os.makedirs('figures', exist_ok=True)
os.makedirs('metrics', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('splits/split_indices.npz'):
    print('Syncing splits from Google Drive...')
    !cp -r /content/drive/MyDrive/ann_cw1/splits/* splits/

# Sync metrics and checkpoints from Drive as well
!cp -r /content/drive/MyDrive/ann_cw1/metrics/* metrics/ 2>/dev/null || true
!cp -r /content/drive/MyDrive/ann_cw1/checkpoints/* checkpoints/ 2>/dev/null || true

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data   = np.load('/content/drive/MyDrive/ann_cw1/eurosat_64.npz')
X, y   = data['X'], data['y']
splits = np.load('splits/split_indices.npz')
train_idx, val_idx, test_idx = splits['train'], splits['val'], splits['test']
with open('splits/preprocessing.json') as f:
    prep = json.load(f)
MEAN = np.array(prep['mean']); STD = np.array(prep['std']); CLASSES = prep['classes']
print('Setup complete.')

## Load all results

In [ ]:
baseline = json.load(open('metrics/baseline_result.json'))
mlp_res  = json.load(open('metrics/mlp_results.json'))
cnn_res  = json.load(open('metrics/cnn_results.json'))

print('=== RESULTS SUMMARY ===')
print(f"{'Model':<15} {'Val F1 (mean±std)':>20} {'Test F1 (mean±std)':>20}")
print('-' * 60)
print(f"{'Baseline':.<15} {'N/A (single run)':>20} {baseline['test_macro_f1']:>10.4f}  {'N/A':>9}")
print(f"{'MLP':.<15} {mlp_res['val_mean']:>7.4f}±{mlp_res['val_std']:>.4f}      {mlp_res['test_mean']:>7.4f}±{mlp_res['test_std']:>.4f}")
print(f"{'CNN':.<15} {cnn_res['val_mean']:>7.4f}±{cnn_res['val_std']:>.4f}      {cnn_res['test_mean']:>7.4f}±{cnn_res['test_std']:>.4f}")

## T4b Per-class metrics — CNN (best seed)

In [ ]:
def conv_block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                         nn.ReLU(inplace=True), nn.MaxPool2d(2))
def make_cnn(p=0.3):
    return nn.Sequential(conv_block(3,32), conv_block(32,64), conv_block(64,128),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p), nn.Linear(128,10))

class EuroSATDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, idx):
        self.X=X[idx]; self.y=y[idx]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        img = (torch.tensor(self.X[i], dtype=torch.float32)/255.0 - torch.tensor(MEAN)) / torch.tensor(STD)
        return img.permute(2,0,1), int(self.y[i])

test_loader = torch.utils.data.DataLoader(EuroSATDataset(X,y,test_idx), batch_size=64, num_workers=2)

best_seed = max(cnn_res['seeds'], key=lambda r: r['val_macro_f1'])['seed']
cnn = make_cnn().to(DEVICE)
cnn.load_state_dict(torch.load(f'checkpoints/cnn_seed{best_seed}.pt'))
cnn.eval()
preds, trues, confs = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        probs = torch.softmax(cnn(xb.to(DEVICE)), dim=1)
        conf, pred = probs.max(1)
        preds.extend(pred.cpu().numpy())
        trues.extend(yb.numpy())
        confs.extend(conf.cpu().numpy())

y_true = np.array(trues); y_pred = np.array(preds); y_conf = np.array(confs)
print(classification_report(y_true, y_pred, target_names=CLASSES))

## T4d Error Analysis — 5 Misclassified Images

In [ ]:
wrong_idx  = np.where(y_true != y_pred)[0]
sample_idx = wrong_idx[:5]
test_images = X[test_idx]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, i in zip(axes, sample_idx):
    ax.imshow(test_images[i])
    ax.axis('off')
    ax.set_title(f'True: {CLASSES[y_true[i]]}\nPred: {CLASSES[y_pred[i]]}\nConf: {y_conf[i]:.2f}', fontsize=7)
plt.suptitle('CNN — Misclassified Test Examples', fontsize=12)
plt.tight_layout()
plt.savefig('figures/error_analysis_cnn.png', dpi=150)
!cp figures/error_analysis_cnn.png /content/drive/MyDrive/ann_cw1/figures/ 2>/dev/null || true
plt.show()
print('Saved: figures/error_analysis_cnn.png (backed up to Drive)')
print(f'Total misclassified: {len(wrong_idx)} / {len(y_true)}')

## T4d Stress Test — Gaussian Noise

**Protocol (written before running):** Add zero-mean Gaussian noise with σ ∈ {0.05, 0.10, 0.20} to normalised test images. Evaluate CNN (best seed) on all three noise levels. Report macro-F1 per sigma. Limitation: this tests sensitivity to synthetic noise, NOT to real atmospheric effects (clouds, haze, sensor noise).

In [ ]:
sigmas = [0.0, 0.05, 0.10, 0.20]
stress_results = []

for sigma in sigmas:
    preds_s = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb + sigma * torch.randn_like(xb)  # add Gaussian noise
            pred = cnn(xb.to(DEVICE)).argmax(1).cpu().numpy()
            preds_s.extend(pred)
    f1 = f1_score(y_true, preds_s, average='macro')
    stress_results.append({'sigma': sigma, 'macro_f1': round(f1, 4)})
    print(f'sigma={sigma:.2f}  macro-F1={f1:.4f}')

stress_data = {'model': 'cnn', 'seed': best_seed, 'stress_test': stress_results}
with open('metrics/stress_test.json', 'w') as f:
    json.dump(stress_data, f, indent=2)
!cp metrics/stress_test.json /content/drive/MyDrive/ann_cw1/metrics/ 2>/dev/null || true
print('Saved: metrics/stress_test.json (backed up to Drive)')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r['sigma'] for r in stress_results], [r['macro_f1'] for r in stress_results],
        'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Gaussian Noise σ', fontsize=12)
ax.set_ylabel('Macro-F1', fontsize=12)
ax.set_title('CNN Robustness — Stress Test (Gaussian Noise)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/stress_test.png', dpi=150)
!cp figures/stress_test.png /content/drive/MyDrive/ann_cw1/figures/ 2>/dev/null || true
plt.show()
print('Saved: figures/stress_test.png (backed up to Drive)')

## Build master results.csv (T6b)

In [ ]:
rows = []
rows.append({'model':'baseline_hog_lr','config':'baseline.yaml','seed':42,
             'val_macro_f1':baseline['val_macro_f1'],'test_macro_f1':baseline['test_macro_f1'],
             'accuracy':baseline['accuracy'],'params':baseline['params'],'inference_ms':baseline['inference_ms']})
for r in mlp_res['seeds']:
    rows.append({'model':'mlp_pytorch','config':'mlp.yaml','seed':r['seed'],
                 'val_macro_f1':round(r['val_macro_f1'],4),'test_macro_f1':round(r['test_macro_f1'],4),
                 'accuracy':round(r.get('val_accuracy',0),4),'params':6425610,'inference_ms':0})
for r in cnn_res['seeds']:
    rows.append({'model':'cnn_pytorch','config':'cnn.yaml','seed':r['seed'],
                 'val_macro_f1':round(r['val_macro_f1'],4),'test_macro_f1':round(r['test_macro_f1'],4),
                 'accuracy':round(r.get('val_accuracy',0),4),'params':94730,'inference_ms':0})

with open('metrics/results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)

!cp metrics/results.csv /content/drive/MyDrive/ann_cw1/metrics/ 2>/dev/null || true
print('Saved: metrics/results.csv (backed up to Drive)')
for r in rows:
    print(r)

---
## ✅ Notebook 07 Complete
**Next:** Run `08_recommendation_inference.ipynb` for T5 model recommendation and inference tool.